In [2]:
# =========================
# PART 1
# =========================

import os
import glob
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

TARGET_FRUITS = ['Apple', 'Banana', 'Strawberry', 'Tomato']

VISION_DATA_DIR = '/kaggle/input/datasets/ulnnproject/food-freshness-dataset/Dataset'
SENSOR_DATA_DIR = '/kaggle/input/datasets/mehrabmahdian/food-freshness-electronic-nose-data/AllSmaples-Report'

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 8

SENSOR_COLUMNS = ['MQ2','MQ3','MQ4','MQ5','MQ6','MQ7','MQ8','MQ9','MQ135']


def get_labels_from_day(day_int):
    days_left = max(0, 5 - day_int)
    is_fresh = 1 if day_int <= 2 else 0
    return is_fresh, days_left


def process_sensor_file(filepath):
    try:
        df = pd.read_csv(filepath)
        df.columns = df.columns.str.strip()
        df_sensors = df[SENSOR_COLUMNS]

        means = df_sensors.mean().values
        variances = df_sensors.var().values

        gradients = np.gradient(df_sensors.values, axis=0)
        mean_gradients = np.mean(gradients, axis=0)

        features = np.concatenate([means, variances, mean_gradients])
        return np.nan_to_num(features)

    except:
        return np.zeros(len(SENSOR_COLUMNS)*3)


class HybridDataGenerator(Sequence):

    def __init__(self, vision_dir, sensor_dir, fruits, batch_size=8, img_size=(224,224), split='train'):
        super().__init__()
        self.vision_dir = vision_dir
        self.sensor_dir = sensor_dir
        self.fruits = fruits
        self.batch_size = batch_size
        self.img_size = img_size

        self.samples = self._build_sample_list()

        random.shuffle(self.samples)
        split_idx = int(len(self.samples)*0.8)

        if split == 'train':
            self.samples = self.samples[:split_idx]
        else:
            self.samples = self.samples[split_idx:]

        self.sensor_cache = {}
        for _, sensor_path, _, _ in self.samples:
            if sensor_path not in self.sensor_cache:
                self.sensor_cache[sensor_path] = process_sensor_file(sensor_path)


    def _build_sample_list(self):
        samples = []
        sensor_files = glob.glob(os.path.join(self.sensor_dir, '* D*.csv'))

        folder_map = {}
        for h_folder in os.listdir(self.vision_dir):
            h_path = os.path.join(self.vision_dir, h_folder)
            if os.path.isdir(h_path):
                state = 'fresh' if 'fresh' in h_folder.lower() else 'rotten'

                for f_folder in os.listdir(h_path):
                    f_path = os.path.join(h_path, f_folder)
                    if os.path.isdir(f_path):
                        clean = f_folder.lower().replace(' ','').replace('s','')

                        for fruit in self.fruits:
                            if fruit.lower() in clean:
                                folder_map[f"{state}_{fruit.lower()}"] = f_path

        for s_file in sensor_files:
            fname = os.path.basename(s_file)

            fruit_name = None
            for f in self.fruits:
                if f.lower() in fname.lower():
                    fruit_name = f
                    break

            if not fruit_name:
                continue

            try:
                day = int(fname.split(' D')[1].split('.csv')[0])
            except:
                continue

            is_fresh, days_left = get_labels_from_day(day)
            state = 'fresh' if is_fresh==1 else 'rotten'

            img_folder = folder_map.get(f"{state}_{fruit_name.lower()}")

            if img_folder:
                images = glob.glob(os.path.join(img_folder, '*.*'))
                for img in images:
                    samples.append((img, s_file, is_fresh, days_left))

                    #  BALANCE FIX (duplicate rotten)
                    if is_fresh == 0:
                        samples.append((img, s_file, is_fresh, days_left))

        return samples


    def __len__(self):
        return min(200, int(len(self.samples)/self.batch_size))


    def __getitem__(self, idx):

        batch = self.samples[idx*self.batch_size:(idx+1)*self.batch_size]

        X_img, X_sensor, y_c, y_r = [], [], [], []

        for img_path, sensor_path, is_fresh, days_left in batch:

            img = load_img(img_path, target_size=self.img_size)
            img = img_to_array(img)

            if random.random() > 0.7:
                img = tf.image.random_flip_left_right(img)

            img = preprocess_input(img)

            X_img.append(img)
            X_sensor.append(self.sensor_cache[sensor_path])
            y_c.append(is_fresh)
            y_r.append(days_left)

        return (np.array(X_img), np.array(X_sensor)), {
            'classification_out': np.array(y_c),
            'regression_out': np.array(y_r)
        }

In [5]:
# =========================
# PART 2
# =========================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Concatenate, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_hybrid_model():

    vision_input = Input(shape=(224,224,3))
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=vision_input)

    #  Fine-tuning last layers
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        layer.trainable = True

    vision_features = GlobalAveragePooling2D()(base_model.output)

    sensor_input = Input(shape=(27,))
    x = Dense(64, activation='relu')(sensor_input)
    x = BatchNormalization()(x)

    x = Dense(32, activation='relu')(x)
    x = BatchNormalization()(x)

    sensor_features = Dense(16, activation='relu')(x)

    fused = Concatenate()([vision_features, sensor_features])
    fused = Dropout(0.5)(fused)

    class_out = Dense(1, activation='sigmoid', name='classification_out')(fused)
    reg_out = Dense(1, activation='linear', name='regression_out')(fused)

    model = Model(inputs=[vision_input, sensor_input], outputs=[class_out, reg_out])

    model.compile(
        optimizer=Adam(0.0005),
        loss={'classification_out':'binary_crossentropy','regression_out':'mse'},
        loss_weights={'classification_out':2.0,'regression_out':0.5},  #  bias fix
        metrics={'classification_out':'accuracy','regression_out':'mae'}
    )

    return model

In [6]:
# =========================
# PART 3
# =========================

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

def train_model(epochs=10):

    train_gen = HybridDataGenerator(VISION_DATA_DIR, SENSOR_DATA_DIR, TARGET_FRUITS, BATCH_SIZE, split='train')
    val_gen = HybridDataGenerator(VISION_DATA_DIR, SENSOR_DATA_DIR, TARGET_FRUITS, BATCH_SIZE, split='val')

    model = build_hybrid_model()

    callbacks = [
        ModelCheckpoint('best_hybrid_fruit_model.h5', save_best_only=True, monitor='val_loss'),
        EarlyStopping(patience=4, restore_best_weights=True),
        ReduceLROnPlateau(patience=2)
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks
    )

    model.save('/kaggle/working/final_hybrid_fruit_model.h5')

    return model, history


if __name__ == "__main__":
    print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
    trained_model, train_history = train_model(epochs=10)

Num GPUs Available: 1


/tmp/ipykernel_55/3787211063.py:13: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=vision_input)
I0000 00:00:1777388785.095920      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10


I0000 00:00:1777388799.710165     123 service.cc:152] XLA service 0x78a2940039b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777388799.710199     123 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1777388801.560845     123 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777388810.473622     123 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - classification_out_accuracy: 0.8367 - classification_out_loss: 0.3604 - loss: 2.0475 - regression_out_loss: 2.6536 - regression_out_mae: 1.2582

...

200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 94ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0093 - loss: 0.1847 - regression_out_loss: 0.3324 - regression_out_mae: 0.4610 - val_classification_out_accuracy: 0.9669 - val_classification_out_loss: 0.1300 - val_loss: 0.9892 - val_regression_out_loss: 1.4585 - val_regression_out_mae: 0.9925 - learning_rate: 5.0000e-05
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0076 - loss: 0.1550 - regression_out_loss: 0.2797 - regression_out_mae: 0.4302

200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0076 - loss: 0.1550 - regression_out_loss: 0.2797 - regression_out_mae: 0.4302 - val_classification_out_accuracy: 0.9706 - val_classification_out_loss: 0.1068 - val_loss: 0.8210 - val_regression_out_loss: 1.2149 - val_regression_out_mae: 0.8829 - learning_rate: 5.0000e-05
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0068 - loss: 0.1441 - regression_out_loss: 0.2610 - regression_out_mae: 0.4050

200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 94ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0068 - loss: 0.1441 - regression_out_loss: 0.2610 - regression_out_mae: 0.4050 - val_classification_out_accuracy: 0.9775 - val_classification_out_loss: 0.0859 - val_loss: 0.6050 - val_regression_out_loss: 0.8665 - val_regression_out_mae: 0.7365 - learning_rate: 5.0000e-05
Epoch 9/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0046 - loss: 0.1344 - regression_out_loss: 0.2506 - regression_out_mae: 0.3967

200/200 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0046 - loss: 0.1345 - regression_out_loss: 0.2507 - regression_out_mae: 0.3967 - val_classification_out_accuracy: 0.9781 - val_classification_out_loss: 0.0828 - val_loss: 0.5090 - val_regression_out_loss: 0.6867 - val_regression_out_mae: 0.6497 - learning_rate: 5.0000e-05
Epoch 10/10
199/200 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0043 - loss: 0.1274 - regression_out_loss: 0.2376 - regression_out_mae: 0.3829

200/200 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - classification_out_accuracy: 1.0000 - classification_out_loss: 0.0043 - loss: 0.1273 - regression_out_loss: 0.2375 - regression_out_mae: 0.3828 - val_classification_out_accuracy: 0.9781 - val_classification_out_loss: 0.0773 - val_loss: 0.4314 - val_regression_out_loss: 0.5537 - val_regression_out_mae: 0.5819 - learning_rate: 5.0000e-05
